# 08 — IC & hypotheses on **final 2024** only

Shortlist frozen from selection. Primary inference: moving-block bootstrap IC (rolling IC is for stability plots, not stars).


In [ ]:
from spy_alpha_research.data import (
    load_config, project_root, load_frame, save_frame, split_three_way, make_target,
)
ROOT = project_root()
exp = load_config(ROOT / "configs" / "experiment.yaml")
models = load_config(ROOT / "configs" / "models.yaml")
bt = load_config(ROOT / "configs" / "backtest.yaml")
FIG = ROOT / exp["paths"]["figures_dir"]
FIG.mkdir(parents=True, exist_ok=True)
PROC = ROOT / exp["paths"]["processed_dir"]
dates = exp["dates"]

from spy_alpha_research.attribution import load_alpha_candidates
from spy_alpha_research.alpha import build_alpha_signals, equal_weight_blend
from spy_alpha_research.statistics import (
    ic_table, bh_fdr, rolling_ic, test_mean_ic_positive, test_combo_ic_positive, test_combo_vs_best,
)
from spy_alpha_research.reporting import plot_ic_bars
import matplotlib.pyplot as plt

feats = load_frame(PROC / "features_raw.parquet")
target = load_frame(PROC / "spy_target.parquet").iloc[:, 0]
train_idx, sel_idx, final_idx = split_three_way(
    feats.index,
    train_end=dates["train_end"],
    selection_start=dates["selection_start"],
    selection_end=dates["selection_end"],
    final_start=dates["final_start"],
    end=dates["end"],
)
alphas = load_alpha_candidates(ROOT / exp["paths"]["alpha_candidates"])
signals = build_alpha_signals(
    feats, alphas, target, train_idx, min_periods=exp["scaling"]["min_periods"]
)
eqw = equal_weight_blend(signals)
all_sigs = {**signals, "eqw_alpha": eqw}
final_sigs = {k: v.reindex(final_idx) for k, v in all_sigs.items()}
table = ic_table(
    final_sigs, target.reindex(final_idx),
    window=exp["statistics"]["rolling_ic_window"],
    nw_lags=exp["alpha"]["nw_lags"],
    n_bootstrap=exp["alpha"]["n_bootstrap"],
    block_size=exp["statistics"]["block_bootstrap_size"],
    seed=exp["project"]["seed"],
)
fdr = bh_fdr(table["p_value"], alpha=exp["alpha"]["fdr_alpha"])
plot_ic_bars(table, FIG / "ic_by_alpha.png")

# Stability plot only (no inference stars)
fig, ax = plt.subplots(figsize=(10, 3))
for name, sig in list(final_sigs.items())[:4]:
    rolling_ic(sig, target.reindex(final_idx), window=exp["statistics"]["rolling_ic_window"]).plot(
        ax=ax, label=name, alpha=0.8
    )
ax.axhline(0, color="black", lw=0.8)
ax.legend(fontsize=7)
ax.set_title("Rolling IC (stability only — not primary significance)")
fig.savefig(FIG / "rolling_ic_stability.png", dpi=150, bbox_inches="tight")
plt.close(fig)
table.join(fdr[["p_adjusted", "reject"]])


In [ ]:
kwargs = dict(
    n_bootstrap=exp["alpha"]["n_bootstrap"],
    block_size=exp["statistics"]["block_bootstrap_size"],
    seed=exp["project"]["seed"],
)
h1 = {k: test_mean_ic_positive(v.reindex(final_idx), target.reindex(final_idx), **kwargs) for k, v in signals.items()}
h2 = test_combo_ic_positive(eqw.reindex(final_idx), target.reindex(final_idx), **kwargs)
h3 = test_combo_vs_best(
    eqw.reindex(final_idx),
    {k: v.reindex(final_idx) for k, v in signals.items()},
    target.reindex(final_idx),
    **kwargs,
)
h1, h2, h3
